In [65]:
#Author: Logan Kline
    # GLBM
    #April 23, 2024
    #Feed in S123 files, search for site updates, then update feature class
    ######################################################################

# UPDATED v3 5.1.2024
#This version addresses AGOL issues with datetime conversions
# UPDATED v4 5.23.2024
#This version addresses includes edits to adapt to BOEM-CLB project, by Mike Bollinger

import arcpy
import pandas as pd
from IPython.display import display
from arcgis.features import GeoAccessor, GeoSeriesAccessor, FeatureLayerCollection
from arcgis.gis import GIS
from arcgis import geometry
import os
import datetime as dt
from datetime import timedelta

sys.path.append(r"C:\Users\Mike.Bollinger\Documents\Python\VehicleTools") ##UPDATE## to Folder where "OverwiteFS.py is located"
import OverwriteFS

gis = GIS("home")
print("Logged in to " + str(gis.properties.name) + " as " + str(gis.properties.user.username))

Logged in to NOAA GeoPlatform as mike.bollinger_noaa


In [66]:
#access and save layers from AGOL
s123_cruise = gis.content.get('00d6b4deec944f3992c7a6ce016f3a7d').layers[0] #NCCOS UMS database cruise level layer, point
s123_auv = gis.content.get('00d6b4deec944f3992c7a6ce016f3a7d').layers[1] #NCCOS UMS AUV database point layer
s123_rov = gis.content.get('00d6b4deec944f3992c7a6ce016f3a7d').layers[2] #NCCOS UMS ROV database point layer
s123_dc = gis.content.get('00d6b4deec944f3992c7a6ce016f3a7d').layers[3] #NCCOS UMS Drop Camera database point layer

boem_samplesites = gis.content.get('29407067efe9494e9d5c49b7afbbdccb').layers[0] #sampling points already up in AGOL


#convert to databases
s123_cruise_dbf = pd.DataFrame(s123_cruise.query().sdf)
s123_auv_dbf = pd.DataFrame(s123_auv.query().sdf)
s123_rov_dbf = pd.DataFrame(s123_rov.query().sdf)
s123_dc_dbf = pd.DataFrame(s123_dc.query().sdf)
boem_samplesites = gis.content.get('29407067efe9494e9d5c49b7afbbdccb').layers[0] #sampling points already up in AGOL
boem_samplesites_dbf = pd.DataFrame(boem_samplesites.query().sdf)


print(boem_samplesites_dbf)

     fid AUV_Completed_Date ROV_Completed_Date DropCam_Completed_Date  \
0      1                NaT                NaT                    NaT   
1      2                NaT                NaT                    NaT   
2      3                NaT                NaT                    NaT   
3      4                NaT                NaT                    NaT   
4      5                NaT                NaT                    NaT   
..   ...                ...                ...                    ...   
145  146                NaT                NaT                    NaT   
146  147                NaT                NaT                    NaT   
147  148                NaT                NaT                    NaT   
148  149                NaT                NaT                    NaT   
149  150                NaT                NaT                    NaT   

    Most_Recent_Survey_Date Status  \
0                       NaT   <NA>   
1                       NaT   <NA>   
2        

In [67]:
if not s123_cruise_dbf.empty:
    # Convert 'entryDay' column to datetime type
    s123_cruise_dbf['entryDay'] = pd.to_datetime(s123_cruise_dbf['entryDay'], errors='coerce')
    four_hours = timedelta(hours=4)
    s123_cruise_dbf['entryDay'] += four_hours

    for index, row in s123_cruise_dbf.iterrows():
        # Check if all specified columns contain NaN values
        if pd.isna(row['EODsuccessfulROV']) and pd.isna(row['EODsuccessfulAUV']) and pd.isna(row['EODsuccessfulDC']):
            continue  # Skip to next iteration if all specified columns contain NaN values

    # Handle non-string values in 'EODsuccessfulAUV' column
        if pd.notna(row['EODsuccessfulAUV']):
            # Extract integers from the string
            integers = [int(x.strip()) for x in str(row['EODsuccessfulAUV']).split(',') if x.strip().isdigit()]

            # Check if integers exist in boem_samplesites_dbf['fid']
            for integer in integers:
                if integer in boem_samplesites_dbf['fid'].values:
                    # Update corresponding ROV_Completed_Date with entryDay value
                    boem_samplesites_dbf.loc[boem_samplesites_dbf['fid'] == integer, 'AUV_Completed_Date'] = row['entryDay']

         # Handle non-string values in 'EODsuccessfulROV' column
        if pd.notna(row['EODsuccessfulROV']):
            # Extract integers from the string
            integers = [int(x.strip()) for x in str(row['EODsuccessfulROV']).split(',') if x.strip().isdigit()]

            # Check if integers exist in boem_samplesites_dbf['fid']
            for integer in integers:
                if integer in boem_samplesites_dbf['fid'].values:
                    # Update corresponding ROV_Completed_Date with entryDay value
                    boem_samplesites_dbf.loc[boem_samplesites_dbf['fid'] == integer, 'ROV_Completed_Date'] = row['entryDay']

        # Handle non-string values in 'EODsuccessfulDC' column
        if pd.notna(row['EODsuccessfulDC']):
            # Extract integers from the string
            integers = [int(x.strip()) for x in str(row['EODsuccessfulDC']).split(',') if x.strip().isdigit()]

            # Check if integers exist in boem_samplesites_dbf['fid']
            for integer in integers:
                if integer in boem_samplesites_dbf['fid'].values:
                    # Update corresponding DropCam_Completed_Date with entryDay value
                    boem_samplesites_dbf.loc[boem_samplesites_dbf['fid'] == integer, 'DropCam_Completed_Date'] = row['entryDay']

else:
    print("NOAA S123 empty, skipping")


def check_status(row):
    if row['AUV_Completed_Date'] < pd.Timestamp(2024, 1, 1):
        return 'Incomplete'
    if row['ROV_Completed_Date'] < pd.Timestamp(2024, 1, 1):
        return 'Incomplete'
    if row['DropCam_Completed_Date'] < pd.Timestamp(2024, 1, 1):
        return 'Incomplete'
    if pd.isna(row['AUV_Completed_Date']) and pd.isna(row['ROV_Completed_Date']) and pd.isna(row['DropCam_Completed_Date']):
        return 'Incomplete'
    #if row['AUV'] == ' ' and row['ROV'] == ' ' and row['DropCam'] == ' ' and row['Ponar'] == ' ':
    #    return 'Extra'
    return 'Complete'

# Apply function to each row to update 'Status'
boem_samplesites_dbf['Status'] = boem_samplesites_dbf.apply(check_status, axis=1)

def check_incomplete(df, columns):
    # Check for NaT in specified columns
    if df[columns].isna().any().any():
        return "Incomplete"



# Columns to check
columns_to_check = ['AUV_Completed_Date', 'ROV_Completed_Date', 'DropCam_Completed_Date']

# Apply the function
result = check_incomplete(boem_samplesites_dbf, columns_to_check)
print(result)

print("Status updated")

#update Most_Recent_Survey_Date
for index, row in boem_samplesites_dbf.iterrows():
    # Extract dates from specified columns
    dates = [row['AUV_Completed_Date'], row['ROV_Completed_Date'], row['DropCam_Completed_Date']]

    # Filter out NaT values
    valid_dates = [date for date in dates if pd.notna(date)]

    # Check if there are any valid dates
    if valid_dates:
        # Get the maximum date
        most_recent_date = max(valid_dates)

        # Update 'Most_Recent_Survey_Date' column with the most recent date
        boem_samplesites_dbf.at[index, 'Most_Recent_Survey_Date'] = most_recent_date

print("Time updated successfully")
print(boem_samplesites_dbf)

Incomplete
Status updated
Time updated successfully
     fid  AUV_Completed_Date  ROV_Completed_Date DropCam_Completed_Date  \
0      1 2024-05-14 04:00:00 2024-05-16 04:00:00                    NaT   
1      2 2024-05-15 04:00:00 2024-05-16 04:00:00                    NaT   
2      3 2024-05-15 04:00:00                 NaT                    NaT   
3      4 2024-05-15 04:00:00                 NaT                    NaT   
4      5                 NaT                 NaT                    NaT   
..   ...                 ...                 ...                    ...   
145  146                 NaT                 NaT                    NaT   
146  147                 NaT                 NaT                    NaT   
147  148                 NaT                 NaT                    NaT   
148  149                 NaT                 NaT                    NaT   
149  150                 NaT                 NaT                    NaT   

    Most_Recent_Survey_Date      Status  \
0   

In [68]:
####start to update feature layer
ss_fset = boem_samplesites.query()

#inner join
overlap_rows = ss_fset.sdf.merge(boem_samplesites_dbf, how='inner', on='fid')
features_for_update = []
ss_features = ss_fset.features

features_for_update_TEST = []
# Create a dictionary for quick lookup of features by SiteName
features_dict = {f.attributes['fid']: f for f in ss_features}

# Vectorized operation to fetch all matching rows at once
matching_rows = boem_samplesites_dbf.loc[boem_samplesites_dbf.fid.isin(overlap_rows['fid'])]

# Prepare attribute updates
attribute_updates = {
    'AUV_Completed_Date': matching_rows['AUV_Completed_Date'],
    'ROV_Completed_Date': matching_rows['ROV_Completed_Date'],
    'DropCam_Completed_Date': matching_rows['DropCam_Completed_Date'],
    'Most_Recent_Survey_Date': matching_rows['Most_Recent_Survey_Date'],
    'Status': matching_rows['Status']
}


# Update features in bulk
for fid, matching_row in matching_rows.iterrows():
    original_feature = features_dict[fid]

# Update attributes in bulk
for attr_name, attr_values in attribute_updates.items():
    for feature, attr_value in zip(ss_features, attr_values):
        feature.attributes[attr_name] = attr_value

print("Attributes updated")

# Add all updated features to the list
features_for_update_TEST.extend(ss_features)

#publish feature layer
boem_samplesites.edit_features(updates= features_for_update_TEST)

KeyError: 0

Logan's Code

In [ ]:

#access and save layers from AGOL
wcsnms_noaa_s123 = gis.content.get('580c143c04e44ebc8ab2cfaead21e469').layers[0] #GLBM database cruise level layer, point
wscnms_usgs_s123 = gis.content.get('ee2a35f781ff42e4a293e2628a86fd1c').layers[0] #USGS AUV database point layer
wscnms_usgs_tracklines = gis.content.get('24a5c128bac24224945718cccacc7710').layers[0] #USGS tracklines with site IDs, polyline
wscnms_samplesites = gis.content.get('1102c34b67a7407690ef53c102e13625').layers[0] #sampling points for field season, points"

#convert to databases
boem_samplesites_dbf = pd.DataFrame(wscnms_samplesites.query().sdf)
wcsnms_noaa_s123_dbf = pd.DataFrame(wcsnms_noaa_s123.query().sdf)
wscnms_usgs_s123_dbf = pd.DataFrame(wscnms_usgs_s123.query().sdf)
wscnms_usgs_tracklines_dbf = pd.DataFrame(wscnms_usgs_tracklines.query().sdf)

if not wcsnms_noaa_s123_dbf.empty:
    # Convert 'entryDay' column to datetime type
    wcsnms_noaa_s123_dbf['entryDay'] = pd.to_datetime(wcsnms_noaa_s123_dbf['entryDay'], errors='coerce')
    four_hours = timedelta(hours=4)
    wcsnms_noaa_s123_dbf['entryDay'] += four_hours

    for index, row in wcsnms_noaa_s123_dbf.iterrows():
        # Check if all specified columns contain NaN values
        if pd.isna(row['EODsuccessfulROV']) and pd.isna(row['EODsuccessfulAUV']) and pd.isna(row['EODsuccessfulDC']) and pd.isna(row['EODsuccessfulPonar']):
            continue  # Skip to next iteration if all specified columns contain NaN values

    # Handle non-string values in 'EODsuccessfulAUV' column
        if pd.notna(row['EODsuccessfulAUV']):
            # Extract integers from the string
            integers = [int(x.strip()) for x in str(row['EODsuccessfulAUV']).split(',') if x.strip().isdigit()]

            # Check if integers exist in wscnms_samplesites_dbf['SiteName']
            for integer in integers:
                if integer in wscnms_samplesites_dbf['SiteName'].values:
                    # Update corresponding ROV_Completed_Date with entryDay value
                    wscnms_samplesites_dbf.loc[wscnms_samplesites_dbf['SiteName'] == integer, 'AUV_Completed_Date'] = row['entryDay']

         # Handle non-string values in 'EODsuccessfulROV' column
        if pd.notna(row['EODsuccessfulROV']):
            # Extract integers from the string
            integers = [int(x.strip()) for x in str(row['EODsuccessfulROV']).split(',') if x.strip().isdigit()]

            # Check if integers exist in wscnms_samplesites_dbf['SiteName']
            for integer in integers:
                if integer in wscnms_samplesites_dbf['SiteName'].values:
                    # Update corresponding ROV_Completed_Date with entryDay value
                    wscnms_samplesites_dbf.loc[wscnms_samplesites_dbf['SiteName'] == integer, 'ROV_Completed_Date'] = row['entryDay']

        # Handle non-string values in 'EODsuccessfulDC' column
        if pd.notna(row['EODsuccessfulDC']):
            # Extract integers from the string
            integers = [int(x.strip()) for x in str(row['EODsuccessfulDC']).split(',') if x.strip().isdigit()]

            # Check if integers exist in wscnms_samplesites_dbf['SiteName']
            for integer in integers:
                if integer in wscnms_samplesites_dbf['SiteName'].values:
                    # Update corresponding DropCam_Completed_Date with entryDay value
                    wscnms_samplesites_dbf.loc[wscnms_samplesites_dbf['SiteName'] == integer, 'DropCam_Completed_Date'] = row['entryDay']

        # Handle non-string values in 'EODsuccessfulPonar' column
        if pd.notna(row['EODsuccessfulPonar']):
            # Extract integers from the string
            integers = [int(x.strip()) for x in str(row['EODsuccessfulPonar']).split(',') if x.strip().isdigit()]

            # Check if integers exist in wscnms_samplesites_dbf['SiteName']
            for integer in integers:
                if integer in wscnms_samplesites_dbf['SiteName'].values:
                    # Update corresponding Ponar_Completed_Date with entryDay value
                    wscnms_samplesites_dbf.loc[wscnms_samplesites_dbf['SiteName'] == integer, 'Ponar_Completed_Date'] = row['entryDay']
else:
    print("NOAA S123 empty, skipping")

wscnms_usgs_s123_dbf = wscnms_usgs_s123_dbf[wscnms_usgs_s123_dbf['name'].str.contains("Shipwreck Coast") & ~wscnms_usgs_s123_dbf['name'].str.contains("Failed") & ~wscnms_usgs_s123_dbf['name'].str.contains("Attempt")]

wscnms_usgs_s123_dbf['date'] = pd.to_datetime(wscnms_usgs_s123_dbf['date'], errors='coerce')
four_hours = timedelta(hours=4)
wscnms_usgs_s123_dbf['date'] += four_hours

#only look at relevant Shipwreck Coast entries from USGS S123

# Iterate through rows
for index, row in wscnms_usgs_s123_dbf.iterrows():
    # Extract number from the string in 'name'
    number = ''.join(filter(str.isdigit, row['name']))
    if not number:  # Check if 'number' is empty after removing non-numeric characters
        continue  # Skip to next iteration if empty

    # Convert 'number' to integer
    number = int(number)

    # Find corresponding row in wscnms_usgs_tracklines_dbf
    trackline_row = wscnms_usgs_tracklines_dbf[wscnms_usgs_tracklines_dbf['MissionNum'] == number]

    if not trackline_row.empty:
        # Extract integers from SampledSit
        sampled_sit = list(map(int, trackline_row['SampledSit'].iloc[0].split(',')))

        # Update AUV_Completed_Date in wscnms_samplesites_dbf
       # date_part = row['date'].date()  # Extracting only the date portion
        #formatted_date = date_part.strftime('%m-%d-%Y %H:%M')  # Formatting as MM-DD-YYYY
        #date_part=+ four_hours
        wscnms_samplesites_dbf.loc[wscnms_samplesites_dbf['SiteName'].isin(sampled_sit), 'AUV_Completed_Date'] = row['date']

print("Table updated")

def check_status(row):
    if 'Yes' in row['AUV'] and row['AUV_Completed_Date'] < pd.Timestamp(2024, 1, 1):
        return 'Incomplete'
    if 'Yes' in row['ROV'] and row['ROV_Completed_Date'] < pd.Timestamp(2024, 1, 1):
        return 'Incomplete'
    if 'Yes' in row['DropCam'] and row['DropCam_Completed_Date'] < pd.Timestamp(2024, 1, 1):
        return 'Incomplete'
    if 'Yes' in row['Ponar'] and row['Ponar_Completed_Date'] < pd.Timestamp(2024, 1, 1):
        return 'Incomplete'
    if row['AUV'] == ' ' and row['ROV'] == ' ' and row['DropCam'] == ' ' and row['Ponar'] == ' ':
        return 'Extra'
    return 'Complete'

# Apply function to each row to update 'Status'
wscnms_samplesites_dbf['Status'] = wscnms_samplesites_dbf.apply(check_status, axis=1)

print("Status updated")

#update Most_Recent_Survey_Date
for index, row in wscnms_samplesites_dbf.iterrows():
    # Extract dates from specified columns
    dates = [row['AUV_Completed_Date'], row['ROV_Completed_Date'], row['DropCam_Completed_Date'], row['Ponar_Completed_Date']]

    # Filter out NaT values
    valid_dates = [date for date in dates if pd.notna(date)]

    # Check if there are any valid dates
    if valid_dates:
        # Get the maximum date
        most_recent_date = max(valid_dates)

        # Update 'Most_Recent_Survey_Date' column with the most recent date
        wscnms_samplesites_dbf.at[index, 'Most_Recent_Survey_Date'] = most_recent_date

print("Time updated successfully")

####start to update feature layer
ss_fset = wscnms_samplesites.query()

#inner join
overlap_rows = ss_fset.sdf.merge(wscnms_samplesites_dbf, how='inner', on='SiteName')
features_for_update = []
ss_features = ss_fset.features

features_for_update_TEST = []
# Create a dictionary for quick lookup of features by SiteName
features_dict = {f.attributes['SiteName']: f for f in ss_features}

# Vectorized operation to fetch all matching rows at once
matching_rows = wscnms_samplesites_dbf.loc[wscnms_samplesites_dbf.SiteName.isin(overlap_rows['SiteName'])]

# Prepare attribute updates
attribute_updates = {
    'AUV_Completed_Date': matching_rows['AUV_Completed_Date'],
    'ROV_Completed_Date': matching_rows['ROV_Completed_Date'],
    'DropCam_Completed_Date': matching_rows['DropCam_Completed_Date'],
    'Most_Recent_Survey_Date': matching_rows['Most_Recent_Survey_Date'],
    'Status': matching_rows['Status']
}

# Update features in bulk
for SiteName, matching_row in matching_rows.iterrows():
    original_feature = features_dict[SiteName]

# Update attributes in bulk
for attr_name, attr_values in attribute_updates.items():
    for feature, attr_value in zip(ss_features, attr_values):
        feature.attributes[attr_name] = attr_value

print("Attributes updated")

# Add all updated features to the list
features_for_update_TEST.extend(ss_features)

#publish feature layer
wscnms_samplesites.edit_features(updates= features_for_update_TEST)

In [ ]:
###troubleshooting
